In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans
from category_encoders import TargetEncoder
import warnings

warnings.filterwarnings('ignore')

print("\n--- 🎇 SON KURŞUN: FRANKENSTEIN VERİ SETİ & MİKRO-ÖĞRENME ---")

# 1. VERİ YÜKLEME (Gerçek IBM Verisi Dahil)
train = pd.read_csv("../data/raw/train.csv", index_col='id')
test = pd.read_csv("../data/raw/test.csv", index_col='id')
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig = pd.read_csv(url).rename(columns={'customerID': 'id'}).set_index('id')

y_train = train['Churn'].map({'Yes': 1, 'No': 0})
train.drop('Churn', axis=1, inplace=True)
y_orig = orig['Churn'].map({'Yes': 1, 'No': 0})
orig.drop('Churn', axis=1, inplace=True)

X_train_full = pd.concat([train, orig])
y_train_full = pd.concat([y_train, y_orig])
df_all = pd.concat([X_train_full, test], keys=['train', 'test'])

# Sıfırıncı ay düzeltmesi
df_all['TotalCharges'] = pd.to_numeric(df_all['TotalCharges'], errors='coerce')
df_all.loc[df_all['tenure'] == 0, 'TotalCharges'] = df_all['MonthlyCharges']
df_all['TotalCharges'].fillna(0, inplace=True)

# -------------------------------------------------------------------
# 2. ŞAMPİYON ÖZELLİKLERİN BİRLEŞİMİ (Frankenstein)
# -------------------------------------------------------------------
# A. Magic Features (Sihirli Özellikler)
df_all['TotalCharges_Diff'] = df_all['TotalCharges'] - (df_all['MonthlyCharges'] * df_all['tenure'])
contract_map = {'Month-to-month': 1, 'One year': 12, 'Two year': 24}
df_all['Contract_Months'] = df_all['Contract'].map(contract_map)
df_all['Tenure_Contract_Ratio'] = df_all['tenure'] / df_all['Contract_Months']
ek_hizmetler = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df_all['Total_Extra_Services'] = (df_all[ek_hizmetler] == 'Yes').sum(axis=1)

# B. Z-Score / Group Stats (Grup Sapmaları)
df_all['Micro_Segment_Billing'] = df_all['Contract'] + "_" + df_all['PaymentMethod']
group_mean = df_all.groupby('Micro_Segment_Billing')['MonthlyCharges'].transform('mean')
group_std = df_all.groupby('Micro_Segment_Billing')['MonthlyCharges'].transform('std').fillna(1)
df_all['Z_Score_MonthlyCharges'] = (df_all['MonthlyCharges'] - group_mean) / group_std

# C. K-Means Clustering (Gözetimsiz Segmentasyon)
kmeans_features = ['tenure', 'MonthlyCharges', 'TotalCharges_Diff']
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_all['Financial_Segment'] = kmeans.fit_predict(df_all[kmeans_features]).astype(str)

# Kategorikleri ayırıyoruz (Target Encoding uygulanacak)
cat_cols = df_all.select_dtypes(include=['object']).columns.tolist()

X_train_final = df_all.xs('train')
X_test_final = df_all.xs('test')

print(f"En iyi özellikler harmanlandı! Toplam {X_train_final.shape[1]} sütun.\n")

# -------------------------------------------------------------------
# 3. MİKRO-ÖĞRENME İLE XGBOOST EĞİTİMİ (5-Fold Target Encoding)
# -------------------------------------------------------------------
# -------------------------------------------------------------------
# 3. MİKRO-ÖĞRENME İLE XGBOOST EĞİTİMİ (5-Fold Target Encoding)
# -------------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_final))
test_preds = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx].copy(), y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx].copy(), y_train_full.iloc[valid_idx]
    X_te_test = X_test_final.copy()
    
    # Tüm kategoriklere izole Target Encoding
    encoder = TargetEncoder(cols=cat_cols, smoothing=10)
    X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols], y_tr)
    X_va[cat_cols] = encoder.transform(X_va[cat_cols])
    X_te_test[cat_cols] = encoder.transform(X_te_test[cat_cols])
    
    # Çok daha yavaş öğrenme (lr=0.015) ve daha ince ayarlar
    model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.015,
        max_depth=5,            
        min_child_weight=3,     
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc",
        early_stopping_rounds=50  # <-- DÜZELTME: PARAMETRE BURAYA GELDİ!
    )
    
    # DÜZELTME: fit içinden early_stopping_rounds silindi
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model.predict_proba(X_va)[:, 1]
    oof_preds[valid_idx] = valid_preds
    test_preds += model.predict_proba(X_te_test)[:, 1] / skf.n_splits
    
    print(f"Frankenstein Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

cv_auc = roc_auc_score(y_train_full, oof_preds)
print(f"\n🏆 Frankenstein (OOF) AUC Skoru: {cv_auc:.4f}")

# Gönderim Dosyası
submission = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds})
submission.to_csv("../submissions/submission_frankenstein_final.csv", index=False)
print("✅ Günün son kurşunu 'submissions' klasörüne kaydedildi!")


--- 🎇 SON KURŞUN: FRANKENSTEIN VERİ SETİ & MİKRO-ÖĞRENME ---
En iyi özellikler harmanlandı! Toplam 26 sütun.

Frankenstein Fold 1 AUC: 0.9154
Frankenstein Fold 2 AUC: 0.9159
Frankenstein Fold 3 AUC: 0.9136
Frankenstein Fold 4 AUC: 0.9148
Frankenstein Fold 5 AUC: 0.9157

🏆 Frankenstein (OOF) AUC Skoru: 0.9151
✅ Günün son kurşunu 'submissions' klasörüne kaydedildi!
